# 🌍 → 🌕 / 🔴  TerraX
A LightGBM model that predicts whether a place on Earth is a **Moon analog**, a **Mars analog**, or **neither**.

Space agencies train astronauts and test rovers at places on Earth that resemble the Moon or Mars, such as Meteor Crater and the Atacama Desert. This notebook learns what those places have in common from NASA data, then scores **any** coordinate, for example `Moon 0.62 · Mars 0.25 · Neither 0.13`.

| Section | Content |
|---|---|
| 1 | Setup |
| 2 | Labelled places (`data/sites.csv`) |
| 3 | Data: NASA imagery, terrain, climate, surface temperature, PDS spectra |
| 4 | Features |
| 5 | Look at the data |
| 6 | Model (LightGBM) |
| 7 | Evaluation on unseen places |
| 8 | Feature importance |
| 9 | Predict new places |
| 10 | Export the model for the website (`webapp/`) |

---
## 1. Setup
**Libraries:** `requests` (web APIs), `numpy` and `pandas`, `Pillow` (images), `lightgbm` (model), `scikit-learn` (evaluation), `matplotlib` (charts).

Every download is cached in `data/cache/`, so re-runs are fast.

In [ ]:
import json, time, re
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from PIL import Image
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix, classification_report

In [ ]:
PATCH_HALF_DEG  = 0.10     # area around each place: +/-0.1 deg (~ +/-11 km)
PATCH_PX        = 48       # image is 48 x 48 pixels (~460 m / pixel)
PIXELS_PER_SITE = 200      # training pixels sampled per place
SEED            = 42

CLASSES = ["Moon", "Mars", "Neither"]                       # class 0, 1, 2
COLORS  = {"Moon": "#2a78d6", "Mars": "#eb6834", "Neither": "#1baf7a"}   # fixed colour per class

DATA = Path("..") / "data"; DATA.mkdir(exist_ok=True)       # the notebook lives in model/, the data one level up
CACHE = DATA / "cache"; CACHE.mkdir(exist_ok=True)
HEADERS = {"User-Agent": "terrax/0.1"}
rng = np.random.default_rng(SEED)

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#888", "axes.labelcolor": "#333", "xtick.color": "#555",
                     "ytick.color": "#555", "axes.grid": True, "grid.color": "#eee"})

---
## 2. Labelled places
| Class | What it means |
|---|---|
| **Moon** | Apollo/Artemis training and lunar test sites: impact craters, fresh basalt lava, cinder fields |
| **Mars** | Mars rover tests, crewed simulations and astrobiology sites: hyper-arid, cold, salty or acidic deserts |
| **Neither** | ordinary land: forests, cities, farmland, wetlands, and dry steppe and scrub (so "dry" alone isn't enough) |

The places are read from **`data/sites.csv`**, with the columns `site, lat, lon, label, region, source`. The labels come from the ESA/Open University *Catalogue of Planetary Analogues* (Preston et al., 2012) and the USGS record of Apollo astronaut field training (Open-File Report 2005-1190, Table 1), plus NASA SSERVI and ESA PANGAEA training sites.

**Spatial groups:** places closer than 150 km share a `group`. Testing never trains on a place and then tests on its near neighbour.

In [ ]:
from scipy.sparse.csgraph import connected_components

sites = pd.read_csv(DATA / "sites.csv")

# spatial groups: link every pair of places closer than 150 km, then take the connected clusters
lat, lon = np.radians(sites.lat.values), np.radians(sites.lon.values)
dlat, dlon = lat[:, None] - lat[None, :], lon[:, None] - lon[None, :]
dist_km = 6371 * 2 * np.arcsin(np.sqrt(np.sin(dlat / 2) ** 2 + np.cos(lat[:, None]) * np.cos(lat[None, :]) * np.sin(dlon / 2) ** 2))
_, sites["group"] = connected_components(dist_km < 150, directed=False)

print(sites.label.value_counts().to_string())
sites.head()

---
## 3. Data
| Source | What it gives | Why it matters |
|---|---|---|
| **Blue Marble** (NASA GIBS, 500 m) | true surface colour | red iron oxide (Mars), dark basalt (Moon), green vegetation (Neither) |
| **ASTER DEM shaded relief** (GIBS) | terrain shape | craters, lava and canyons are rough |
| **NASA POWER** climatology | temperature, rainfall, humidity, sunlight, wind, elevation | analogs are dry, with harsh temperature swings |
| **PDS RELAB** spectra (3d) | lab spectra of Moon and Mars samples | the real colour of the Moon and Mars |
| **MODIS surface temperature** (GIBS, 3e) | daytime ground temperature in two seasons | how strongly the ground heats and cools |
| **Macrostrat geologic maps** (3f) | rock type | basalt and impact rock (Moon-like) vs sediments and salts (Mars-like) |

GIBS is the image service behind [Worldview](https://worldview.earthdata.nasa.gov), so every layer can also be viewed there.

In [ ]:
# resources
GIBS_WMS = "https://gibs.earthdata.nasa.gov/wms/epsg4326/best/wms.cgi"
POWER    = "https://power.larc.nasa.gov/api/temporal/climatology/point"
RELAB = "https://pds-geosciences.wustl.edu/speclib/urn-nasa-pds-relab/data_reflectance"
LST_COLORMAP_URL = "https://gibs.earthdata.nasa.gov/colormaps/v1.3/MODIS_Land_Surface_Temp.xml"
MACROSTRAT = "https://macrostrat.org/api/v2/geologic_units/map"

In [ ]:

POWER_PARAMS = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "ALLSKY_SFC_SW_DWN", "WS2M"]

def _get(url, params=None, expect=None, tries=4):
    # GET with simple retry (APIs sometimes time out).
    # expect = required start of the Content-Type (e.g. "image"), so an error page is never saved as data.
    for i in range(tries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=60)
            if r.status_code == 200 and (expect is None or r.headers.get("Content-Type", "").startswith(expect)):
                return r
        except requests.RequestException:
            pass
        time.sleep(2 ** i)
    raise RuntimeError(f"failed: {url} {params}")

def gibs_patch(layer, lat, lon, half=PATCH_HALF_DEG, px=PATCH_PX, date=None):
    # Download a small square image (px x px) of a GIBS layer centred on lat/lon.
    # date: for layers that change over time, e.g. "2023-07-01" (None = the layer's default).
    # Returns float array (px, px, 4) = R,G,B,Alpha in 0..1. Cached on disk.
    tag = f"_{date}" if date else ""
    f = CACHE / f"{layer}{tag}_{lat:.3f}_{lon:.3f}_{half}_{px}.png"
    if not f.exists():
        params = dict(SERVICE="WMS", REQUEST="GetMap", VERSION="1.3.0", LAYERS=layer,
                      CRS="EPSG:4326", STYLES="", FORMAT="image/png", WIDTH=px, HEIGHT=px,
                      BBOX=f"{lat-half},{lon-half},{lat+half},{lon+half}")   # WMS 1.3: lat,lon order
        if date:
            params["TIME"] = date
        f.write_bytes(_get(GIBS_WMS, params, expect="image").content)
    return np.asarray(Image.open(f).convert("RGBA"), dtype=np.float32) / 255.0

def power_climate(lat, lon):
    # Long-term annual climate averages from NASA POWER (+ elevation). Cached on disk.
    f = CACHE / f"power_{lat:.3f}_{lon:.3f}.json"
    if not f.exists():
        r = _get(POWER, dict(parameters=",".join(POWER_PARAMS), community="AG",
                             latitude=lat, longitude=lon, format="JSON"), expect="application/json")
        f.write_text(r.text)
    d = json.loads(f.read_text())
    p = d["properties"]["parameter"]
    out = {k: (np.nan if p[k]["ANN"] == -999 else p[k]["ANN"]) for k in POWER_PARAMS}
    out["elevation_m"] = d["geometry"]["coordinates"][2]
    return out

### Look at a few places
Colour (top row) and terrain relief (bottom row) for two places per class.

In [ ]:
show = ["Meteor Crater, Arizona", "Craters of the Moon, Idaho", "Atacama Desert (Yungay), Chile",
        "Wadi Rum, Jordan", "Amazon rainforest, Brazil", "Iowa farmland, USA"]

fig, axes = plt.subplots(2, len(show), figsize=(2.2 * len(show), 4.8))
for j, name in enumerate(show):
    s = sites.set_index("site").loc[name]
    c = gibs_patch("BlueMarble_NextGeneration", s.lat, s.lon)
    h = gibs_patch("ASTER_GDEM_Greyscale_Shaded_Relief", s.lat, s.lon)

    axes[0, j].imshow(np.clip(c[..., :3] * 1.8, 0, 1))      # brightened for display only
    axes[1, j].imshow(h[..., 0], cmap="gray")
    axes[0, j].set_title(f"{name.split(',')[0]}\n[{s.label}]", fontsize=8, color="#222")
    for a in axes[:, j]: a.set_xticks([]); a.set_yticks([]); a.grid(False)

axes[0, 0].set_ylabel("colour", fontsize=9); axes[1, 0].set_ylabel("relief", fontsize=9)
plt.tight_layout(); plt.show()

---
### PDS reference spectra (RELAB)
The PDS Geosciences Node hosts **RELAB**, a library of laboratory reflectance spectra. A spectrum shows how much light a material reflects at each wavelength.

| Reference | Samples |
|---|---|
| `moon_soil` | 14 Apollo soils: dark mare and bright highland |
| `mars_dust` | 5 × JSC Mars-1, NASA's simulant of red Martian dust |
| `mars_rock` | 7 Martian meteorites |

Each spectrum is averaged over the three MODIS bands that Blue Marble is built from (blue, green and red). Its colour **shape** is then described by two numbers: `slope_rg = ln(R/G)` and `slope_gb = ln(G/B)`. Every Earth pixel gets the same two numbers. Blue Marble is a display image, so it's first converted back to approximate reflectance (by undoing the sRGB gamma).

In [ ]:
# folder/spectrum_id : sample name (from the PDS label)
PDS_SPECTRA = {
    "moon_soil": {
        "bdr2/c1lr15": "Apollo 11 mare soil 10084",   "bdr2/c1lr10": "Apollo 12 mare soil 12001",
        "bdr2/c1lr30": "Apollo 12 mare soil 12030",   "bdr3/c2lr80": "Apollo 15 mare soil 15071",
        "bdr2/c1lr23": "Apollo 17 mare soil 70181",   "bdr2/c1lr28": "Apollo 17 mare soil 71501",
        "bdr3/c1lr41": "Apollo 17 mare soil 79221",   "bdr3/c1lr59": "Apollo 17 mare soil 71061",
        "bdr3/c1lr104": "Apollo 16 highland soil 62231", "bdr3/c1lr112": "Apollo 16 highland soil 61141",
        "bdr3/c1ls110": "Apollo 16 highland soil 64421", "bdr3/c1ls115": "Apollo 16 highland soil 67461",
        "bdr2/c1ls128": "Apollo 16 highland soil 67511", "bdr3/c1lr89": "Apollo 14 soil 14163",
    },
    "mars_dust": {
        "bdr2/c1jb239a": "JSC Mars-1 (A)", "bdr2/c1jb240a": "JSC Mars-1 (B)", "bdr3/c1jb521a": "JSC Mars-1 palagonite",
        "bdr3/c1jbh86a": "JSC Mars-1 <125 um", "bdr3/c1jb569a": "JSC Mars-1 (C)",
    },
    "mars_rock": {
        "bdr3/c1mt223p": "Tissint", "bdr3/c1mt224p": "NWA 1068", "bdr3/c1mt225p": "NWA 2626",
        "bdr3/c1mt226p": "NWA 6234", "bdr3/c1mt227p": "NWA 6963", "bdr3/c1mt228p": "NWA 7397",
        "bdr3/c1mt04a": "EETA79001",
    },
}
MODIS_BANDS = {"R": (620, 670), "G": (545, 565), "B": (459, 479)}   # MODIS bands 1, 4, 3 (nm)

def relab_spectrum(path):
    # Download one RELAB spectrum (.tab) from the PDS. Returns wavelength (nm), reflectance (0..1).
    f = CACHE / f"relab_{path.replace('/', '_')}.tab"
    if not f.exists():
        f.write_text(_get(f"{RELAB}/{path}.tab").text)
    lines = f.read_text().splitlines()
    n = int(lines[0].split()[0])                       # first line = number of data rows
    rows = [l.split()[:2] for l in lines[1:] if len(l.split()) >= 2][:n]
    a = np.array(rows, dtype=float)
    return a[:, 0], a[:, 1]

def band_values(wl, refl):
    # average reflectance inside each MODIS band -> [R, G, B]
    return np.array([refl[(wl >= lo) & (wl <= hi)].mean() for lo, hi in MODIS_BANDS.values()])

def colour_shape(rgb):
    # brightness-independent colour shape: [ln(R/G), ln(G/B)]
    return np.array([np.log(rgb[0] / rgb[1]), np.log(rgb[1] / rgb[2])])

def srgb_to_linear(v):
    # undo the standard display brightness curve (sRGB gamma) -> roughly proportional to reflectance
    return np.where(v <= 0.04045, v / 12.92, ((v + 0.055) / 1.055) ** 2.4)

REF_RGB   = {k: np.mean([band_values(*relab_spectrum(p)) for p in d], axis=0) for k, d in PDS_SPECTRA.items()}
REF_SHAPE = {k: colour_shape(v) for k, v in REF_RGB.items()}
pd.DataFrame({k: {"red": v[0], "green": v[1], "blue": v[2], "red/green": v[0] / v[1], "green/blue": v[1] / v[2]}
              for k, v in REF_RGB.items()}).T.round(3)

In [ ]:
STYLE = {"moon_soil": dict(color=COLORS["Moon"], ls="-"), "mars_dust": dict(color=COLORS["Mars"], ls="-"),
         "mars_rock": dict(color=COLORS["Mars"], ls="--")}
fig, ax = plt.subplots(figsize=(8, 4))
for name, (lo, hi) in MODIS_BANDS.items():
    ax.axvspan(lo, hi, color="#e6e6e6", zorder=0)
    ax.text((lo + hi) / 2, 0.58, name, ha="center", fontsize=9, color="#555")
for k, d in PDS_SPECTRA.items():
    for j, p in enumerate(d):
        wl, refl = relab_spectrum(p)
        keep = (wl >= 350) & (wl <= 1000)
        ax.plot(wl[keep], refl[keep], lw=0.8, alpha=0.6, label=k if j == 0 else None, **STYLE[k])
ax.set_xlim(350, 1000); ax.set_ylim(0, 0.62)
ax.set_xlabel("wavelength (nm)"); ax.set_ylabel("reflectance")
ax.set_title("PDS / RELAB spectra used as planetary references", fontsize=10)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

---
### Surface temperature
Dry, bare ground heats and cools far more than vegetated or wet ground; LRO Diviner measured this behaviour on the Moon. On Earth we use **MODIS land surface temperature**: monthly daytime maps for January and July 2023 from GIBS. GIBS serves them as coloured images, so each colour is converted back to kelvin using the published colormap.

| Feature | Meaning |
|---|---|
| `lst_season` | July − January daytime ground temperature (K): how strongly the ground heats and cools over the year |


In [ ]:
LST_MONTHS = ["2023-01-01", "2023-07-01"]

cmf = CACHE / "MODIS_Land_Surface_Temp_colormap.xml"
if not cmf.exists():
    cmf.write_text(_get(LST_COLORMAP_URL).text)
entries = re.findall(r'rgb="(\d+),(\d+),(\d+)"[^>]*?value="\[([\d.]+),([\d.]+)\)"', cmf.read_text())
CM_RGB = np.array([[int(r), int(g), int(b)] for r, g, b, _, _ in entries], dtype=np.float32) / 255
CM_K   = np.array([(float(lo) + float(hi)) / 2 for *_, lo, hi in entries])
CM_K[0], CM_K[-1] = 200.0, 350.0          # the first/last bins are open-ended (<200 K, >350 K)

def decode_lst(img):
    # colour image (px,px,4) -> temperature in kelvin (px,px); NaN where there is no data
    rgb = img[..., :3].reshape(-1, 1, 3)
    dist = ((rgb - CM_RGB[None]) ** 2).sum(-1)               # distance to every colormap colour
    k = CM_K[dist.argmin(1)].reshape(img.shape[:2])          # nearest colour -> its temperature
    nodata = (img[..., 3] == 0) | (dist.min(1).reshape(img.shape[:2]) > 0.003)
    return np.where(nodata, np.nan, k)

def lst_season(lat, lon):
    # July - January daytime ground temperature (K), per pixel
    jan, jul = [decode_lst(gibs_patch("MODIS_Terra_L3_Land_Surface_Temp_Monthly_Day", lat, lon, date=m)) for m in LST_MONTHS]
    return np.abs(jul - jan)                                 # abs: the seasons are reversed in the south

---
### Rock type (geologic maps)
The rock under a place is the most direct link to the Moon or Mars: basalt and impact rock are lunar-like, while sediments, salts and clays point to Mars's watery past. **[Macrostrat](https://macrostrat.org)** (University of Wisconsin) combines the geologic maps of national geological surveys into one free API. Several maps can cover the same point and sometimes disagree, so for the centre of each place every map unit returned **votes** (more detailed maps get more votes). Each rock description is sorted into one of 10 classes, for example `mafic_volcanic` (basalt), `evaporite` (salt, gypsum), `clastic_sediment` (sandstone, shale) or `impact`.

The feature `rock` is **categorical**, and LightGBM handles categories directly.

In [ ]:
# rock classes, checked in this order; the first class with a matching keyword wins
ROCK_RULES = [
    ("impact",           ["impact", "breccia", "suevite"]),
    ("evaporite",        ["evaporite", "salt", "gypsum", "halite", "playa", "saline", "sabkha", "sebkha"]),
    ("mafic_volcanic",   ["basalt", "basaltic", "mafic volcanic", "lava"]),
    ("felsic_volcanic",  ["rhyolite", "dacite", "tuff", "ignimbrite", "andesite", "felsic volcanic", "pyroclastic"]),
    ("volcanic",         ["volcanic"]),
    ("intrusive",        ["granite", "gabbro", "anorthosite", "diorite", "intrusive", "plutonic", "granodiorite", "syenite"]),
    ("metamorphic",      ["gneiss", "schist", "metamorphic", "quartzite", "marble", "amphibolite", "greenstone"]),
    ("carbonate",        ["limestone", "dolomite", "carbonate", "chalk", "travertine"]),
    ("unconsolidated",   ["alluvium", "alluvial", "unconsolidated", "sand", "gravel", "dune", "loess", "colluvium", "till", "glacial", "eolian", "aeolian"]),
    ("clastic_sediment", ["sandstone", "shale", "siltstone", "conglomerate", "mudstone", "sedimentary", "siliciclastic", "clastic"]),
]
ROCK_CLASSES = [cls for cls, _ in ROCK_RULES] + ["unknown"]
SCALE_WEIGHT = {"large": 4, "medium": 3, "small": 2, "tiny": 1}     # votes per map unit: more detailed map = more votes

def classify_unit(u):
    # rock class of one map unit (whole words only, so "salt" does not match inside "basalt")
    text = f"{u.get('lith', '')} {u.get('name', '')}".lower()
    for cls, words in ROCK_RULES:
        if any(re.search(r"\b" + re.escape(w) + r"\b", text) for w in words):
            return cls
    return None

def rock_type(lat, lon):
    # Rock class at the centre of a place. Several geologic maps can cover the same point and may disagree,
    # so every map unit votes for its class; more detailed maps get more votes. Cached on disk.
    f = CACHE / f"macrostrat_{lat:.3f}_{lon:.3f}.json"
    if not f.exists():
        f.write_text(_get(MACROSTRAT, dict(lat=lat, lng=lon), expect="application/json").text, encoding="utf-8")
    votes = {}
    for u in json.loads(f.read_text(encoding="utf-8")).get("success", {}).get("data", []):
        cls = classify_unit(u)
        if cls:
            votes[cls] = votes.get(cls, 0) + SCALE_WEIGHT.get(u.get("scale"), 1)
    return max(votes, key=votes.get) if votes else "unknown"

---
## 4. Features
Each pixel (about 460 m) becomes one row of numbers. With 200 pixels per place, about 108 places give about 21,000 rows.

| Features | From | Meaning |
|---|---|---|
| `b`, `brightness` | colour | dark basalt vs bright sand or salt |
| `slope_rg`, `slope_gb` | colour shape (3d) | lunar grey vs Martian red |
| `patch_roughness`, `patch_color_var` | the whole ±11 km area | rugged, patchy terrain |
| `patch_slope_gb` | average of `slope_gb` over the area | the overall colour of the place; steadier than a single pixel |
| `T2M`, `temp_range`, `PRECTOTCORR`, `RH2M`, `ALLSKY_SFC_SW_DWN`, `WS2M`, `elevation_m` | NASA POWER | climate and altitude |
| `lst_season` | surface temperature (3e) | seasonal heating and cooling of the ground |
| `rock` | geologic map (3f) | rock class at the centre of the place (categorical) |

Water pixels are dropped. Latitude and longitude are **not** features, so the model learns what a place is like, not where it is.

In [ ]:
def place_features(lat, lon):
    # Return a DataFrame with one row per land pixel around lat/lon.
    c = gibs_patch("BlueMarble_NextGeneration", lat, lon)                 # colour (48,48,4)
    relief = gibs_patch("ASTER_GDEM_Greyscale_Shaded_Relief", lat, lon)[..., 0]   # terrain (48,48)
    r, g, b, alpha = c[..., 0], c[..., 1], c[..., 2], c[..., 3]
    brightness = (r + g + b) / 3
    lin = srgb_to_linear(c[..., :3]) + 1e-3                  # approx. reflectance; +1e-3 avoids log(0)
    f = {
        "b": b,
        "brightness": brightness,
        "slope_rg": np.log(lin[..., 0] / lin[..., 1]),       # ln(red/green)
        "slope_gb": np.log(lin[..., 1] / lin[..., 2]),       # ln(green/blue)
        "lst_season": lst_season(lat, lon),          # seasonal ground-temperature swing (K)
    }
    df = pd.DataFrame({k: v.ravel() for k, v in f.items()})
    water = (b > r) & (b > g) & (brightness < 0.35)
    df = df[((alpha > 0) & ~water).ravel()].reset_index(drop=True)   # keep land pixels only
    # whole-area context (same value for every pixel of this place)
    df["patch_roughness"] = relief.std()
    df["patch_color_var"] = brightness.std()
    df["patch_slope_gb"] = df["slope_gb"].mean()            # average colour shape of the whole area
    # climate (same value for every pixel of this place)
    clim = power_climate(lat, lon)
    for k, v in clim.items():
        df[k] = v
    df["temp_range"] = clim["T2M_MAX"] - clim["T2M_MIN"]
    # geology (same value for every pixel of this place)
    df["rock"] = pd.Categorical([rock_type(lat, lon)] * len(df), categories=ROCK_CLASSES)
    return df

FEATURES = [
    "b", "brightness", "slope_rg", "slope_gb",                                            # colour
    "patch_roughness", "patch_color_var", "patch_slope_gb",                              # area context
    "T2M", "temp_range", "PRECTOTCORR", "RH2M", "ALLSKY_SFC_SW_DWN", "WS2M", "elevation_m",  # climate
    "lst_season",                                                                         # surface temperature
    "rock",                                                                               # geology (categorical)
]

### Build the training table
For every place: compute the features, keep 200 random land pixels, and add the label and group. The first run downloads about 800 small files (5–10 minutes); later runs use the cache.

In [ ]:
rows = []
for i, s in enumerate(sites.itertuples(), 1):
    df = place_features(s.lat, s.lon)
    if len(df) > PIXELS_PER_SITE:
        df = df.sample(PIXELS_PER_SITE, random_state=SEED)
    df["site"], df["label"], df["group"] = s.site, s.label, s.group
    rows.append(df)
    print(f"\r{i}/{len(sites)} {s.site[:40]:40s} {len(df)} pixels", end="")
    
train = pd.concat(rows, ignore_index=True)
train["y"] = train.label.map({c: i for i, c in enumerate(CLASSES)})
print("\n", train.shape)
train.groupby("label").size()